In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import everything we need from the data loader module
from data_loader import (
    load_all_splits, verify_data,
    CLASSES, SORTED_CLASSES, DATA_DIR,
)

print('Classes (alphabetical, index = label):')
for i, c in enumerate(SORTED_CLASSES):
    print(f'  {i} = {c}')

In [ ]:
verify_data()

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_all_splits(size=64)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.astype(np.float32))
X_val_s   = scaler.transform(X_val.astype(np.float32))
X_test_s  = scaler.transform(X_test.astype(np.float32))

print(f'X_train_s: {X_train_s.shape}, dtype={X_train_s.dtype}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best_f1, best_C, best_clf = -1, None, None

for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # handles 3.3:1 imbalance
        solver='lbfgs',
        C=C,
        n_jobs=-1,
    )
    clf.fit(X_train_s, y_train)
    val_pred = clf.predict(X_val_s)
    acc = (val_pred == y_val).mean()
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f'C={C:<7}  val acc: {acc:.3f}   macro-F1: {f1:.3f}')
    if f1 > best_f1:
        best_f1, best_C, best_clf = f1, C, clf

print(f'\nBest: C={best_C}, val macro-F1={best_f1:.3f}')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = best_clf.predict(X_val_s)

print('Validation classification report:')
print(classification_report(y_val, val_pred,
                            target_names=SORTED_CLASSES, digits=3))

print('\nValidation confusion matrix:')
cm = confusion_matrix(y_val, val_pred)
print(pd.DataFrame(cm, index=SORTED_CLASSES, columns=SORTED_CLASSES))